In [54]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.support.ui import Select
import time
import pandas as pd
import json
import ast

In [3]:
MAIN_URL = "https://www.imdb.com/"

In [29]:
driver = webdriver.Chrome()
driver.get(MAIN_URL)

Get ratings for all episodes of a TV show

In [30]:
search_box = driver.find_element(By.ID, 'suggestion-search')
# or search_box = driver.find_element(By.NAME, 'q')
search_box.clear()
search_box.send_keys("Law & Order: Special Victims Unit")

In [31]:
search_box.submit()

In [33]:
accept = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Accept')]"))).click()

In [34]:
driver.find_element(By.PARTIAL_LINK_TEXT, "Law & Order: Special Victims Unit").click()

In [35]:
episodes = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div/section[1]/div/section/div/div[1]/section[2]/div[1]/a')
driver.get(episodes.get_attribute('href'))

In [36]:
season_links = []
season_elements = driver.find_elements(By.XPATH, '//*[@id="__next"]/main/div/section/div/section/div/div[1]/section[2]/section[1]/div[2]/ul/a')
for season_element in season_elements:
    season_links.append((season_element.text, season_element.get_attribute("href")))

In [37]:
season_links

[('1', 'https://www.imdb.com/title/tt0203259/episodes/?season=1'),
 ('2', 'https://www.imdb.com/title/tt0203259/episodes/?season=2'),
 ('3', 'https://www.imdb.com/title/tt0203259/episodes/?season=3'),
 ('4', 'https://www.imdb.com/title/tt0203259/episodes/?season=4'),
 ('5', 'https://www.imdb.com/title/tt0203259/episodes/?season=5'),
 ('6', 'https://www.imdb.com/title/tt0203259/episodes/?season=6'),
 ('7', 'https://www.imdb.com/title/tt0203259/episodes/?season=7'),
 ('8', 'https://www.imdb.com/title/tt0203259/episodes/?season=8'),
 ('9', 'https://www.imdb.com/title/tt0203259/episodes/?season=9'),
 ('10', 'https://www.imdb.com/title/tt0203259/episodes/?season=10'),
 ('11', 'https://www.imdb.com/title/tt0203259/episodes/?season=11'),
 ('12', 'https://www.imdb.com/title/tt0203259/episodes/?season=12'),
 ('13', 'https://www.imdb.com/title/tt0203259/episodes/?season=13'),
 ('14', 'https://www.imdb.com/title/tt0203259/episodes/?season=14'),
 ('15', 'https://www.imdb.com/title/tt0203259/episod

In [41]:
season_data = []
for season, link in season_links:
    driver.get(link)
    time.sleep(2) 
    
    episodes = {}
    titles_h4 = driver.find_elements(By.TAG_NAME, "h4")
    title_episode = [title.text for title in titles_h4]
    episodes["Title"] = title_episode
    
    ratings = driver.find_elements(By.CLASS_NAME, "ipc-rating-star--rating")
    episodes["Rating"] = [rating.text for rating in ratings[2:]]
    # print(episodes)
    season_data.append(episodes)
    
driver.quit()

In [45]:
df_ratings = pd.DataFrame(season_data)
df_ratings.head()

,Title,Rating
0,"[S1.E1 ∙ Payback, S1.E2 ∙ A Single Life, S1.E3...","[8.2, 8.0, 7.7, 8.1, 7.7, 7.8, 8.3, 8.1, 7.5, ..."
1,"[S2.E1 ∙ Wrong Is Right, S2.E2 ∙ Honor, S2.E3 ...","[8.1, 7.7, 8.6, 8.3, 8.4, 7.8, 7.7, 8.2, 7.8, ..."
2,"[S3.E1 ∙ Repression, S3.E2 ∙ Wrath, S3.E3 ∙ St...","[8.2, 8.3, 8.4, 8.0, 8.1, 7.9, 8.1, 8.0, 8.2, ..."
3,"[S4.E1 ∙ Chameleon, S4.E2 ∙ Deception, S4.E3 ∙...","[8.5, 8.2, 8.0, 7.8, 7.8, 8.5, 8.3, 8.2, 8.4, ..."
4,"[S5.E1 ∙ Tragedy, S5.E2 ∙ Manic, S5.E3 ∙ Mothe...","[8.0, 8.0, 8.2, 9.0, 8.2, 8.6, 7.6, 8.0, 8.3, ..."


In [71]:
df_ratings = df_ratings.iloc[:-1]

In [74]:
df_exploded = df_ratings.explode(["Title", "Rating"]).reset_index(drop=True)
df_exploded["Season"] = df_exploded.index // 22 + 1
df_exploded.to_csv("data/law_and_order_ratings.csv", index=False)

Scrape top 10 box office movies from imdb

In [4]:
driver = webdriver.Chrome()
driver.get(MAIN_URL)

In [8]:
driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
time.sleep(2)
top_box_section = driver.find_element(By.CLASS_NAME, "top-box-office")
link = top_box_section.find_element(By.CLASS_NAME, "ipc-title-link-wrapper").get_attribute("href")
driver.get(link)

In [13]:
movies = driver.find_elements(By.CLASS_NAME, "ipc-title-link-wrapper")
movie_links = [movie.get_attribute("href") for movie in movies if movie.get_attribute("href").startswith("https://www.imdb.com/title/")]
movie_links
assert len(movie_links) == 10, "Incorrect number of movies scraped"

In [23]:
movie_data = []
for link in movie_links:
    print("Scraping movie ", link)
    driver.get(link)
    time.sleep(1)
    movie = {}
    title = driver.find_element(By.TAG_NAME, "h1")
    movie["Title"] = title.text
    print("Title: ", title.text)
    
    # Get the parent of the title element; you can do up to 3 levels with "../../.."
    title_parent = title.find_element(By.XPATH, "..") # it is in the same container as the year and duration
    year_duration = title_parent.find_element(By.TAG_NAME, "ul")
    movie["Year"] = year_duration.find_elements(By.TAG_NAME, "li")[0].text
    print("Year: ", movie["Year"])
    movie["Duration"] = year_duration.find_elements(By.TAG_NAME, "li")[-1].text
    print("Duration: ", movie["Duration"])
    rating = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[2]/div[2]/div/div[1]/a/span/div/div[2]/div[1]').text
    movie["Rating"] = rating.split("\n")[0]
    print("Rating: ", movie["Rating"])
    #tags
    tags = driver.find_elements(By.CLASS_NAME, "ipc-chip__text")
    movie["Tags"] = [tag.text for tag in tags if len(tag.text) > 0]
    print("Tags: ", movie["Tags"])
    #plot
    plot = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[3]/div[2]/div[1]/section/p/span[2]').text
    movie["Plot"] = plot
    print("Plot: ", movie["Plot"])
    
    movie_data.append(movie)

driver.quit()

Scraping movie  https://www.imdb.com/title/tt16366836/?ref_=chtbo_t_1
Title:  Venom: The Last Dance
Year:  2024
Duration:  1h 50m
Rating:  6.2
Tags:  ['Alien Invasion', 'Kaiju', 'Road Trip', 'Superhero', 'Action', 'Adventure', 'Sci-Fi', 'Thriller']
Plot:  Eddie and Venom, on the run, face pursuit from both worlds. As circumstances tighten, they're compelled to make a heart-wrenching choice that could mark the end of their symbiotic partnership.
Scraping movie  https://www.imdb.com/title/tt29623480/?ref_=chtbo_t_2
Title:  The Wild Robot
Year:  2024
Duration:  1h 42m
Rating:  8.3
Tags:  ['Computer Animation', 'Survival', 'Animation', 'Sci-Fi']
Plot:  After a shipwreck, an intelligent robot called Roz is stranded on an uninhabited island. To survive the harsh environment, Roz bonds with the island's animals and cares for an orphaned baby goose.
Scraping movie  https://www.imdb.com/title/tt29268110/?ref_=chtbo_t_3
Title:  Smile 2
Year:  2024
Duration:  2h 7m
Rating:  7.2
Tags:  ['Psycholog

In [26]:
with open("data/top_movies.json", "w") as f:
    json.dump(movie_data, f)

In [27]:
df = pd.DataFrame(movie_data)
df["Tags"] = df["Tags"].apply(lambda x: ", ".join(x))
df.to_csv("data/top_movies.csv", index=False)

In [28]:
df.head()

,Title,Year,Duration,Rating,Tags,Plot
0,Venom: The Last Dance,2024,1h 50m,6.2,"Alien Invasion, Kaiju, Road Trip, Superhero, A...","Eddie and Venom, on the run, face pursuit from..."
1,The Wild Robot,2024,1h 42m,8.3,"Computer Animation, Survival, Animation, Sci-Fi","After a shipwreck, an intelligent robot called..."
2,Smile 2,2024,2h 7m,7.2,"Psychological Horror, Horror, Mystery, Thriller","About to embark on a world tour, global pop se..."
3,Conclave,2024,2h,7.7,"Conspiracy Thriller, Drama, Thriller",When Cardinal Lawrence is tasked with leading ...
4,Here,2024,1h 44m,6.3,"Coming-of-Age, Drama",A generational story about families and the sp...
